# 02 반도체 공정 데이터 분석 · 5. 보강 실습

- 강의 페이지: `Web/강좌/02_반도체_공정_데이터분석/반도체_공정_데이터분석.html` → 목차 **보강 실습**
- 점검 → 정리 → 이상값 확인 → 비교 → 그래프·결론 순서로 직접 분석해 봅니다.
- `반도체_공정_샘플.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

## 초보자 필수 보강 실습

이번에는 `반도체_공정_샘플.csv`를 처음 받았다고 가정하고,
**점검 → 정리 → 비교 → 이상값 확인 → 보고서 작성** 순서로 반복합니다.

숫자가 계산되었다는 사실보다, 그 숫자가 어떤 공정 판단에 필요한지 한 문장으로
설명하는 습관이 중요합니다.


### 준비 · 라이브러리와 숫자 열 목록

In [ ]:
# 📦 분석에 사용할 라이브러리 불러오기
# 'as pd'는 앞으로 pandas를 pd라는 짧은 이름으로 부르겠다는 약속
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 한글 설정 (Mac은 'AppleGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False   # 음수(-) 기호 깨짐 방지

# 실습 4에서 사용할 숫자 센서 열 목록 (3단계와 동일)
numeric_cols = ['온도_섭씨', '압력_Pa', '가스유량_slm', '전력_W',
                '진공도_mTorr', '두께_nm', '습도_pct', '진동_mm_s',
                '처리시간_sec', '냉각수온도_섭씨']


### 실습 1 · 1분 데이터 건강검진

**목표:** 분석 전에 데이터 크기, 결측률, 중복, 판정 비율을 빠르게 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: CSV를 practice_raw로 읽으세요.
# TODO 2: 행·열 수, 열별 결측률(%), 중복 행 수를 출력하세요.
# TODO 3: 합격여부의 건수와 비율(%)을 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

결측률은 `isna().mean().mul(100)`, 판정 비율은 `value_counts(normalize=True)`로 구합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
practice_raw = pd.read_csv('반도체_공정_샘플.csv')

print(f'데이터 크기: {practice_raw.shape[0]}행 x {practice_raw.shape[1]}열')

In [ ]:
print('\n결측률 상위 5개 열(%):')
print(practice_raw.isna().mean().mul(100).sort_values(ascending=False).head().round(2))

In [ ]:
print(f'\n중복 행: {practice_raw.duplicated().sum()}개')

In [ ]:
print('\n판정 건수:')
print(practice_raw['합격여부'].value_counts(dropna=False))

In [ ]:
print('\n판정 비율(%):')
print(practice_raw['합격여부'].value_counts(normalize=True, dropna=False).mul(100).round(2))

### 실습 2 · 재현 가능한 데이터 정리

**목표:** 숫자 결측값을 중앙값으로 채우고 중복을 제거한 분석용 데이터를 만듭니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: practice_raw를 practice_clean으로 복사하세요.
# TODO 2: 숫자 열을 자동으로 찾으세요.
# TODO 3: 각 숫자 열의 결측값을 중앙값으로 채우세요.
# TODO 4: 중복 행을 제거하고 인덱스를 다시 매기세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`select_dtypes(include='number')`로 숫자 열을 자동 선택할 수 있습니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
practice_clean = practice_raw.copy()
practice_numeric = practice_clean.select_dtypes(include='number').columns
practice_clean[practice_numeric] = practice_clean[practice_numeric].fillna(
    practice_clean[practice_numeric].median()
)
practice_clean = practice_clean.drop_duplicates().reset_index(drop=True)
practice_clean['판정'] = practice_clean['합격여부'].map({1: '합격', -1: '불합격'})

print(f'정리 후 크기: {practice_clean.shape}')
print(f'남은 결측값: {practice_clean.isna().sum().sum()}개')


### 실습 3 · IQR로 이상값 후보 찾기

**목표:** 온도 분포에서 일반 범위를 크게 벗어난 측정값을 점검 대상으로 표시합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 온도_섭씨의 Q1(25%)과 Q3(75%)를 구하세요.
# TODO 2: IQR = Q3 - Q1을 계산하세요.
# TODO 3: Q1 - 1.5*IQR 미만 또는 Q3 + 1.5*IQR 초과 행을 찾으세요.
# 주의: 이상값 후보가 반드시 오류나 불량이라는 뜻은 아닙니다.


<details>
<summary><strong>힌트 보기</strong></summary>

`quantile(0.25)`와 `quantile(0.75)`를 사용하고 두 조건은 `|`로 연결합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
q1 = practice_clean['온도_섭씨'].quantile(0.25)
q3 = practice_clean['온도_섭씨'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

temperature_outliers = practice_clean[
    (practice_clean['온도_섭씨'] < lower) |
    (practice_clean['온도_섭씨'] > upper)
]

print(f'온도 일반 범위: {lower:.2f} ~ {upper:.2f} °C')
print(f'이상값 후보: {len(temperature_outliers)}건')
temperature_outliers[['온도_섭씨', '압력_Pa', '두께_nm', '판정']].head()


### 실습 4 · 합격과 불합격의 평균 차이 찾기

**목표:** 두 판정 그룹의 평균 차이가 큰 센서를 우선 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 판정별 숫자 열 평균을 계산하세요.
# TODO 2: 합격 평균과 불합격 평균의 차이 절댓값을 계산하세요.
# TODO 3: 평균 차이가 큰 상위 5개 열을 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

판정별 평균표를 만든 뒤 두 행을 빼고 `.abs().sort_values(ascending=False)`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
compare_cols = [col for col in numeric_cols if col in practice_clean.columns]
result_means = practice_clean.groupby('판정')[compare_cols].mean()
mean_gap = (
    result_means.loc['불합격'] - result_means.loc['합격']
).abs().sort_values(ascending=False)

print('합격/불합격 평균 차이 상위 5개 변수:')
print(mean_gap.head(5).round(3))


### 실습 5 · 핵심 그래프와 한 줄 결론 만들기

**목표:** 판정 건수와 평균 차이가 가장 큰 변수의 분포를 함께 보고 간단히 해석합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 평균 차이가 가장 큰 변수 이름을 top_gap_col에 저장하세요.
# TODO 2: 왼쪽에는 판정별 건수, 오른쪽에는 해당 변수의 박스플롯을 그리세요.
# TODO 3: 합격/불합격 평균과 차이를 한 문장으로 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`mean_gap.index[0]`으로 변수명을 얻고 `plt.subplots(1, 2)`로 그래프 영역을 만듭니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
top_gap_col = mean_gap.index[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=practice_clean, x='판정', hue='판정', legend=False, ax=axes[0])
axes[0].set_title('판정별 건수')

sns.boxplot(data=practice_clean, x='판정', y=top_gap_col,
            hue='판정', legend=False, ax=axes[1])
axes[1].set_title(f'판정별 {top_gap_col} 분포')
plt.tight_layout()
plt.show()

In [ ]:
pass_mean = result_means.loc['합격', top_gap_col]
fail_mean = result_means.loc['불합격', top_gap_col]
print(
    f'{top_gap_col}: 합격 평균 {pass_mean:.2f}, '
    f'불합격 평균 {fail_mean:.2f}, 차이 {abs(fail_mean - pass_mean):.2f}'
)
print('이 차이는 원인 확정이 아니라 추가 점검이 필요한 단서입니다.')

## 마무리

- 건강검진 → 정리 → IQR 이상값 후보 → 평균 차이 → 그래프와 한 줄 결론 순서로 분석했습니다.
- 숫자를 계산한 뒤에는 그 숫자가 어떤 공정 판단에 필요한지 한 문장으로 설명해 보세요.

다음 노트북(`06_AI_미니프로젝트.ipynb`)에서 배운 내용을 AI와 함께 복습하고 응용해 봅니다.